# Thai Character Classification: Complete Master Pipeline
### Solution 3 (Pai + Pooh) — Medical Image & Deep Learning

This master notebook provides an end-to-end implementation for classifying **72 Thai Character Classes** using:
1. **Zero-Leakage Group-Aware Deterministic 80/20 Train-Test Splitting**.
2. **Domain-Specific Glyphic Augmentations** (Letterboxing, Stroke Dilation/Erosion, Bounded Rotations).
3. **Three Comparative Neural Architectures**:
   - `CustomGlyphCNN` (4-Stage Conv-BN-Mish-SE with native $32\times 32$ receptive field)
   - `AdaptedResNet18` (Stem-adapted ImageNet Transfer Learning with Differential Learning Rates)
   - `AdaptedMobileNetV3` (Inverted Residuals with Squeeze-and-Excitation attention)
4. **Class-Balanced Focal Loss** + Label Smoothing ($\epsilon=0.08$) to resolve the severe 1-to-5,025 sample imbalance.
5. **In-Depth Error Analysis**: Confusion Matrices, Top Confused Character Pairs, and Sample Visualizations.

## 1. System Setup & Imports

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn

# Add src to Python Path
src_dir = Path('src').resolve()
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from dataset import build_split_dataframes, create_dataloaders, get_class_weights, letterbox_pad, tis620_to_char
from transforms import get_train_transform, get_val_transform, MorphologicalTransform
from models import build_model
from losses import build_loss_fn
from trainer import ThaiCharacterTrainer, build_optimizer, build_scheduler
from evaluate import evaluate_model_full, plot_confusion_matrix, plot_training_history
from inference import ThaiCharacterInferenceEngine, THAI_CHAR_METADATA

DATASET_DIR = (Path.cwd() / '..' / '..' / 'ThaiCharacter Dataset' / 'round2').resolve()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
print(f'Dataset Path: {DATASET_DIR}')


## 2. Zero-Leakage Deterministic 80/20 Splitting & Verification

In [ ]:
train_df, test_df, class_to_idx = build_split_dataframes(DATASET_DIR, train_ratio=0.8)

total_samples = len(train_df) + len(test_df)
print(f'Total Images: {total_samples:,}')
print(f'Train Set:    {len(train_df):,} samples ({len(train_df)/total_samples*100:.2f}%) | {train_df["class_idx"].nunique()} classes')
print(f'Test Set:     {len(test_df):,} samples ({len(test_df)/total_samples*100:.2f}%) | {test_df["class_idx"].nunique()} classes')

# Data Leakage Verification
train_groups = set(train_df['group_key'].unique())
test_groups = set(test_df['group_key'].unique())
leakage = train_groups.intersection(test_groups)
assert len(leakage) == 0, 'Leakage detected!'
print('✅ Zero Data Leakage verified: No document scans or pages overlap between train and test!')


## 3. Visualizing Preprocessing & Morphological Augmentations

In [ ]:
sample_rows = train_df.sample(8, random_state=42)
morph = MorphologicalTransform(p_dilate=0.5, p_erode=0.5, kernel_size=2)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for i, (_, row) in enumerate(sample_rows.iterrows()):
    with Image.open(row['filepath']) as raw_img:
        padded = letterbox_pad(raw_img.convert('RGB'), target_size=(32, 32))
        augmented = morph(padded)
        
        axes[0, i].imshow(padded)
        axes[0, i].set_title(f"{row['class_number']}: {row['character']}\n(Original)", fontsize=10)
        axes[0, i].axis('off')
        
        axes[1, i].imshow(augmented)
        axes[1, i].set_title(f"{row['character']}\n(Morph Aug)", fontsize=10)
        axes[1, i].axis('off')

plt.suptitle('Aspect-Preserving Letterbox Padding (32x32) & Stroke Morphological Dilation/Erosion', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## 4. Dataloaders with Balanced Sampling & Class Weights

In [ ]:
BATCH_SIZE = 64
TARGET_SIZE = (32, 32)

train_loader, test_loader, _, _, _ = create_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=BATCH_SIZE,
    target_size=TARGET_SIZE,
    train_transform=get_train_transform(max_angle=8.0, p_morphology=0.3),
    test_transform=get_val_transform(),
    use_balanced_sampler=True,
    num_workers=2,
)

num_classes = len(class_to_idx)
class_weights = get_class_weights(train_df, num_classes=num_classes, beta=0.999)
print(f'Train Batches: {len(train_loader)} | Test Batches: {len(test_loader)}')


## 5. Model Architecture Training (CustomGlyphCNN Baseline)

In [ ]:
model_custom = build_model('custom_cnn', num_classes=num_classes, dropout_rate=0.3)
criterion = build_loss_fn('class_balanced_focal', class_weights=class_weights, gamma=2.0, label_smoothing=0.08)
optimizer = build_optimizer(model_custom, optimizer_name='adamw', base_lr=1e-3, weight_decay=1e-2)
scheduler = build_scheduler(optimizer, scheduler_type='cosine_warmup', total_epochs=6, warmup_epochs=2)

trainer = ThaiCharacterTrainer(
    model=model_custom,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    save_dir='checkpoints/custom_cnn',
)

history = trainer.fit(num_epochs=6)
plot_training_history(history)


## 6. Comprehensive Evaluation & Confusion Matrix Analysis

In [ ]:
eval_results = evaluate_model_full(model_custom, test_loader, class_to_idx, device=device)

print(f'\n🏆 CustomGlyphCNN Test Performance:')
print(f'Top-1 Accuracy: {eval_results["top1_accuracy"]*100:.2f}%')
print(f'Top-5 Accuracy: {eval_results["top5_accuracy"]*100:.2f}%')
print(f'Macro F1-Score: {eval_results["macro_f1"]*100:.2f}%')
print(f'Weighted F1:   {eval_results["weighted_f1"]*100:.2f}%')

# Normalized Confusion Matrix Heatmap
plot_confusion_matrix(eval_results['confusion_matrix'], eval_results['idx_to_char'], top_n=30)

# Top Confused Character Pairs
conf_df = pd.DataFrame(eval_results['top_confusions'])
print('\nTop 10 Most Confused Thai Character Pairs:')
display(conf_df[['true_char', 'pred_char', 'count']].head(10))


## 7. Interactive Single Image Predictor

In [ ]:
engine = ThaiCharacterInferenceEngine(DATASET_DIR, device=device)
engine.load_model('custom_cnn', 'checkpoints/custom_cnn/best_model.pt')

# Select a random test image
sample_test_row = test_df.sample(1).iloc[0]
print(f'Testing on Real Ground Truth: {sample_test_row["character"]} (TIS-620: {sample_test_row["class_number"]})')

pred_res = engine.predict(sample_test_row['filepath'], model_name='custom_cnn', top_k=5)
top_p = pred_res['top_prediction']

print(f'\nPredicted: {top_p["character"]} ({top_p["name_th"]} - {top_p["name_en"]})')
print(f'Confidence: {top_p["confidence_pct"]}% | Type: {top_p["type"]}')

print('\nTop 5 Candidates:')
for rank, p in enumerate(pred_res['predictions'], 1):
    print(f"{rank}. {p['character']} ({p['name_th']}) - {p['confidence_pct']:.2f}%")
